# 05 — Graph Viability

Checks whether the proposed bipartite atom ↔ query-point graph is
computationally viable for each protein size (small / medium / large).

## Graph structure

| Node set | Source | Count |
|----------|--------|-------|
| Atoms | PQR file (all atoms, **H included**) | varies |
| Query pts | Surface vertices, curvature-sampled at 5% | varies |

| Edge type | Method | k / cutoff | Feature |
|-----------|--------|-----------|---------|
| Atom–atom (covalent) | MDAnalysis bond guessing | distance-based | bond order scalar {1.0, 1.5, 2.0} + RBF dist |
| Atom–atom (sparse) | kNN per atom, bonds excluded | k = 16 | RBF dist |
| Atom→query | kNN per query (query-centric) | k = 32 | RBF dist |
| Query–query | kNN per query | k = 8 | RBF dist |

All edge counts are **directed**. Bond orders assigned via residue/atom-name heuristic.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.topology.guessers import guess_types
from scipy.spatial import KDTree

warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent.parent))

from src.surface.esp_mapping import curvature_sampling
from src.utils.config import get_data_root
from src.utils.paths import ProteinPaths

In [ ]:
DATA_ROOT = get_data_root()

PROTEIN_IDS = [
    "AF-P01082-F1",   # small   ~7 600 verts
    "AF-Q16613-F1",   # medium  ~30 500 verts
    "AF-B1WC58-F1",   # large   ~215 000 verts
]

SAMPLE_FRAC  = 0.05   # fraction of surface vertices used as query points
KNN_SUPP     = 16     # supplementary atom-atom sparse kNN (bonds excluded)
KNN_AQ       = 32     # atom→query kNN (query-centric: k atoms per query)
KNN_QQ       = 8      # query-query kNN

# RBF distance encoding
N_RBF        = 16     # number of RBF basis functions per edge

# VRAM estimation model
HIDDEN_DIM   = 128    # assumed hidden dimension

print(f"Data root  : {DATA_ROOT}")
print(f"Proteins   : {len(PROTEIN_IDS)}")
print(f"Sampling   : {SAMPLE_FRAC*100:.0f}% of surface vertices")
print(f"kNN        : supp={KNN_SUPP}  AQ={KNN_AQ}  QQ={KNN_QQ}")
print(f"RBF feats  : {N_RBF}  |  hidden dim: {HIDDEN_DIM}")

## Helper functions

In [ ]:
# ── Aromatic ring atom lookup (for bond order heuristic) ──────────────────────
_AROMATIC_ATOMS = {
    "PHE": {"CG", "CD1", "CD2", "CE1", "CE2", "CZ"},
    "TYR": {"CG", "CD1", "CD2", "CE1", "CE2", "CZ"},
    "HIS": {"CG", "ND1", "CD2", "CE1", "NE2"},
    "TRP": {"CG", "CD1", "CD2", "NE1", "CE2", "CE3", "CZ2", "CZ3", "CH2"},
}


def _assign_bond_order(bond) -> float:
    """
    Heuristic bond order for a standard amino-acid bond:
      1.0  — single (default, and all X-H bonds)
      1.5  — aromatic (both atoms in a known ring of PHE/TYR/HIS/TRP)
      2.0  — double (C-O where O has exactly 1 bond total, i.e. terminal carbonyl)
    """
    a1, a2 = bond.atoms
    e1, e2 = a1.element, a2.element

    # X-H bonds are always single
    if "H" in (e1, e2):
        return 1.0

    # Aromatic ring check
    rn = a1.resname
    ring = _AROMATIC_ATOMS.get(rn)
    if ring and a1.name in ring and a2.name in ring:
        return 1.5

    # C=O: terminal oxygen (exactly 1 bond means no H attached → pure carbonyl)
    if {e1, e2} == {"C", "O"}:
        o_atom = a1 if e1 == "O" else a2
        if len(o_atom.bonds) == 1:
            return 2.0

    return 1.0


def build_bond_graph(pqr_path: Path):
    """
    Parse covalent bonds from a PQR file using MDAnalysis bond guessing.
    Elements are guessed from atom names (not present in PQR format).

    Returns:
        atom_xyz   : (N, 3) float32
        atom_names : list[str]
        bond_src   : (B,) int64  — directed: both i→j and j→i included
        bond_dst   : (B,) int64
        bond_orders: (B,) float32  — {1.0, 1.5, 2.0}
        bond_dists : (B,) float32  — Å
        bond_set   : set of (min_i, max_i) undirected pairs  — for exclusion
    """
    # to_guess=['bonds'] is the non-deprecated API in MDAnalysis 2.x
    u = mda.Universe(str(pqr_path), to_guess=["bonds"])
    u.add_TopologyAttr("elements", guess_types(u.atoms.names))

    atom_xyz   = u.atoms.positions.astype(np.float32)
    atom_names = list(u.atoms.names)

    src_list, dst_list, order_list, dist_list = [], [], [], []
    bond_set = set()

    for bond in u.bonds:
        i, j   = bond.atoms[0].index, bond.atoms[1].index
        order  = _assign_bond_order(bond)
        dist   = float(bond.length())
        # directed: both directions
        src_list += [i, j]
        dst_list += [j, i]
        order_list += [order, order]
        dist_list  += [dist,  dist]
        bond_set.add((min(i, j), max(i, j)))

    return (
        atom_xyz,
        atom_names,
        np.array(src_list,   dtype=np.int64),
        np.array(dst_list,   dtype=np.int64),
        np.array(order_list, dtype=np.float32),
        np.array(dist_list,  dtype=np.float32),
        bond_set,
    )

In [ ]:
def knn_edges(xyz: np.ndarray, k: int):
    """
    Directed kNN edges within a single node set (self excluded).
    Returns (src_idx, tgt_idx, dists) each of length N*k.
    """
    tree = KDTree(xyz)
    dists, indices = tree.query(xyz, k=k + 1)   # col 0 is self
    n = len(xyz)
    src_idx = np.repeat(np.arange(n, dtype=np.int64), k)
    tgt_idx = indices[:, 1:].flatten().astype(np.int64)
    d_flat  = dists[:, 1:].flatten().astype(np.float32)
    return src_idx, tgt_idx, d_flat


def knn_edges_bipartite(src_xyz: np.ndarray, tgt_xyz: np.ndarray, k: int):
    """
    For each node in tgt_xyz, find k nearest nodes in src_xyz.
    Returns directed edges src→tgt with (src_idx, tgt_idx, dists).
    Used for atom→query pass: src=atoms, tgt=queries.
    """
    tree = KDTree(src_xyz)
    dists, indices = tree.query(tgt_xyz, k=k)
    n_tgt   = len(tgt_xyz)
    tgt_idx = np.repeat(np.arange(n_tgt, dtype=np.int64), k)
    src_idx = indices.flatten().astype(np.int64)
    d_flat  = dists.flatten().astype(np.float32)
    return src_idx, tgt_idx, d_flat


def knn_supp_edges(atom_xyz: np.ndarray, k: int, bond_set: set):
    """
    kNN=k sparse atom-atom edges, excluding covalent bond pairs.
    Queries k + 8 neighbors per atom (buffer for bond exclusions),
    then filters bond pairs and takes the first k survivors.

    Returns (src_idx, tgt_idx, dists).
    """
    buf  = k + 8   # buffer; max covalent bonds per atom ≈ 4 (heavy) + H
    tree = KDTree(atom_xyz)
    dists_all, idx_all = tree.query(atom_xyz, k=buf + 1)  # +1 for self
    # col 0 is self — skip it
    dists_all = dists_all[:, 1:]
    idx_all   = idx_all[:, 1:]

    src_list, tgt_list, d_list = [], [], []
    for i in range(len(atom_xyz)):
        count = 0
        for rank in range(buf):
            j = int(idx_all[i, rank])
            if (min(i, j), max(i, j)) in bond_set:
                continue   # skip covalent bond
            src_list.append(i)
            tgt_list.append(j)
            d_list.append(float(dists_all[i, rank]))
            count += 1
            if count == k:
                break

    return (
        np.array(src_list, dtype=np.int64),
        np.array(tgt_list, dtype=np.int64),
        np.array(d_list,   dtype=np.float32),
    )


def rbf_encode(dists: np.ndarray, n_rbf: int, d_min: float, d_max: float) -> np.ndarray:
    """
    Encode distances as Gaussian RBF features.
    Centers are evenly spaced in [d_min, d_max]; sigma = spacing.

    Returns:
        (len(dists), n_rbf) float32 array in [0, 1]
    """
    centers = np.linspace(d_min, d_max, n_rbf, dtype=np.float32)
    sigma   = (d_max - d_min) / (n_rbf - 1) if n_rbf > 1 else 1.0
    d       = dists[:, None].astype(np.float32)
    return np.exp(-((d - centers) ** 2) / (sigma ** 2))

## Graph construction — per protein

In [ ]:
rows = []

for pid in PROTEIN_IDS:
    bar = "\u2550" * 68
    print(f"\n{bar}")
    print(f"  {pid}")
    print(f"{bar}")
    t_start = time.perf_counter()
    p = ProteinPaths(pid, DATA_ROOT)

    # ── Covalent bond graph (MDAnalysis) ────────────────────────────────────────
    t0 = time.perf_counter()
    atom_xyz, atom_names, cov_src, cov_dst, cov_orders, cov_dists, bond_set = (
        build_bond_graph(p.pqr_path)
    )
    n_atoms  = len(atom_xyz)
    n_h      = sum(1 for nm in atom_names if nm.startswith("H"))
    n_heavy  = n_atoms - n_h
    n_bonds  = len(bond_set)          # undirected bond count
    n_cov    = len(cov_src)           # directed: 2 × n_bonds
    order_counts = {
        "single"  : int((cov_orders == 1.0).sum() // 2),
        "aromatic": int((cov_orders == 1.5).sum() // 2),
        "double"  : int((cov_orders == 2.0).sum() // 2),
    }
    rbf_cov = rbf_encode(cov_dists, N_RBF, d_min=0.9, d_max=1.8)
    print(
        f"  Atoms      : {n_atoms:>8,}  (heavy: {n_heavy:,}  H: {n_h:,})  [{time.perf_counter()-t0:.2f}s]"
    )
    print(
        f"  Bonds      : {n_bonds:>8,}  (single: {order_counts['single']:,}  "
        f"aromatic: {order_counts['aromatic']:,}  double: {order_counts['double']:,})  "
        f"→ {n_cov:,} directed edges"
    )

    # ── Query points (curvature-sampled surface vertices at 5%) ─────────────────
    t0      = time.perf_counter()
    mesh    = np.load(p.pqr_mesh_path)
    verts   = mesh["verts"]
    faces   = mesh["faces"]
    ses_area = float(mesh["ses_area"])
    k_sample  = max(1, round(SAMPLE_FRAC * len(verts)))
    q_idx     = curvature_sampling(verts, faces, k_sample, ses_area)
    query_xyz = verts[q_idx]
    n_query   = len(query_xyz)
    print(
        f"  Query pts  : {n_query:>8,}  ({SAMPLE_FRAC*100:.0f}% of {len(verts):,} verts)  "
        f"[{time.perf_counter()-t0:.2f}s]"
    )

    # ── Supplementary atom-atom kNN (bonds excluded) ────────────────────────────
    t0 = time.perf_counter()
    supp_src, supp_dst, supp_dists = knn_supp_edges(atom_xyz, KNN_SUPP, bond_set)
    n_supp = len(supp_src)
    rbf_supp = rbf_encode(supp_dists, N_RBF, d_min=1.8, d_max=8.0)
    print(
        f"  Supp edges : {n_supp:>8,}  (kNN={KNN_SUPP}, bonds excl., directed)  "
        f"[{time.perf_counter()-t0:.2f}s]"
    )

    # ── Atom→query edges (kNN=32, query-centric) ────────────────────────────────
    t0 = time.perf_counter()
    aq_src, aq_dst, aq_dists = knn_edges_bipartite(atom_xyz, query_xyz, KNN_AQ)
    n_aq = len(aq_src)
    rbf_aq = rbf_encode(aq_dists, N_RBF, d_min=0.0, d_max=12.0)
    print(
        f"  AQ edges   : {n_aq:>8,}  (kNN={KNN_AQ} atoms/query, directed)  "
        f"[{time.perf_counter()-t0:.2f}s]"
    )

    # ── Query-query edges (kNN=8) ───────────────────────────────────────────────
    t0 = time.perf_counter()
    qq_src, qq_dst, qq_dists = knn_edges(query_xyz, KNN_QQ)
    n_qq = len(qq_src)
    rbf_qq = rbf_encode(qq_dists, N_RBF, d_min=0.0, d_max=8.0)
    print(
        f"  QQ edges   : {n_qq:>8,}  (kNN={KNN_QQ}, directed)  "
        f"[{time.perf_counter()-t0:.2f}s]"
    )

    total_nodes = n_atoms + n_query
    total_edges = n_cov + n_supp + n_aq + n_qq
    elapsed     = time.perf_counter() - t_start

    print(f"\n  \u25ba Nodes : {total_nodes:,}  (atoms={n_atoms:,}  query={n_query:,})")
    print(f"  \u25ba Edges : {total_edges:,}  "
          f"(cov={n_cov:,}  supp={n_supp:,}  AQ={n_aq:,}  QQ={n_qq:,})")
    print(f"  \u25ba Elapsed: {elapsed:.1f}s")

    rows.append({
        "protein"    : pid,
        "n_atoms"    : n_atoms,
        "n_heavy"    : n_heavy,
        "n_H"        : n_h,
        "n_query"    : n_query,
        "cov_edges"  : n_cov,
        "supp_edges" : n_supp,
        "aq_edges"   : n_aq,
        "qq_edges"   : n_qq,
        "total_nodes": total_nodes,
        "total_edges": total_edges,
        "elapsed_s"  : round(elapsed, 1),
    })

## Summary table

In [ ]:
df = pd.DataFrame(rows).set_index("protein")
print("Graph sizes — directed edge counts\n" + "─" * 80)
display(df)

## Notes

### Edge types
- **Covalent edges** — directed (i→j and j→i). Bond order assigned by heuristic:
  - `1.0` single (default, all X–H bonds)
  - `1.5` aromatic (both atoms in PHE/TYR/HIS/TRP ring)
  - `2.0` double (C–O where O has exactly 1 bond, i.e. terminal carbonyl)
- **Supp edges** — kNN=16 per atom with covalent pairs removed; captures medium-range geometry without redundancy.
- **AQ edges** — query-centric: each query receives messages from its 32 nearest atoms. Total = `n_query × 32`.
- **QQ edges** — directed kNN=8; each query has exactly 8 outgoing edges.

### RBF encoding
Each edge carries an N_RBF-dimensional feature vector:
```
centers = linspace(d_min, d_max, N_RBF)
sigma   = spacing between centers
f_k(d)  = exp(-(d - centers_k)^2 / sigma^2)
```
RBF range per edge type: cov [0.9, 1.8 Å], supp [1.8, 8.0 Å], AQ/QQ [0.0, 12.0/8.0 Å].

### Message passing order
| Pass | Edge set | Direction |
|------|----------|-----------|
| 0. Bond pass | cov | i→j |
| 1. Sparse pass | supp | i→j |
| 2. Atom→query | AQ | atom→query |
| 3. Query→query | QQ | i→j |

## VRAM estimates

In [ ]:
def vram_mb(n_nodes, n_edges, hidden_dim, n_rbf, bytes_per_el=4):
    """
    Approximate VRAM for one graph forward pass (float32).

    Accounts for:
      node_emb   — node feature tensors stored in memory
      edge_rbf   — RBF features on every edge (input to MLPs)
      edge_msg   — message tensors aggregated along edges (hidden_dim wide)
    Does NOT include model weights, batch norm buffers, or Python overhead.

    Training (Adam) adds ~4x: model params + grads + 2 optimizer moment tensors
    on top of activations. Reported separately as a multiplier.
    """
    node_emb_mb  = n_nodes * hidden_dim * bytes_per_el / 1e6
    edge_rbf_mb  = n_edges * n_rbf      * bytes_per_el / 1e6
    edge_msg_mb  = n_edges * hidden_dim * bytes_per_el / 1e6
    fwd_mb       = node_emb_mb + edge_rbf_mb + edge_msg_mb
    train_mb     = fwd_mb * 4   # rough: grads + Adam m1/m2 + activations
    return fwd_mb, train_mb


vram_rows = []
for row in rows:
    fwd_mb, train_mb = vram_mb(
        row["total_nodes"], row["total_edges"], HIDDEN_DIM, N_RBF
    )
    vram_rows.append({
        "protein"       : row["protein"],
        "total_nodes"   : row["total_nodes"],
        "total_edges"   : row["total_edges"],
        "fwd_pass_MB"   : round(fwd_mb,   1),
        "train_est_MB"  : round(train_mb, 1),
        "fwd_pass_GB"   : round(fwd_mb   / 1024, 3),
        "train_est_GB"  : round(train_mb / 1024, 3),
    })

vdf = pd.DataFrame(vram_rows).set_index("protein")
print(f"VRAM estimates  (hidden_dim={HIDDEN_DIM}  n_rbf={N_RBF}  float32)\n" + "─" * 68)
display(vdf)
print()
print("Note: 'train_est' ≈ 4× forward pass (grads + Adam states + activations).")
print("      Model weights and Python overhead are NOT included.")